# Implementing Transformers in PyTorch: Architecture and Mechanisms

These notes cover the practical implementation of the Transformer architecture ("Attention is All You Need") using PyTorch. The focus is on the fundamental building blocks, tensor dimensions, mathematical operations, and PyTorch specific mechanisms like `einsum`.

## 1. Core Architecture Overview

The standard Transformer follows an **Encoder-Decoder** architecture.

* **Encoder:** Processes the input sequence into a continuous contextual representation.
* **Decoder:** Generates the output sequence token-by-token, using both its own previous outputs and the Encoder's contextual representation.

**PyTorch Setup Basics:**
Implementations rely heavily on `torch` (for tensor operations) and `torch.nn` (for neural network layers like linear projections and activations). Hardware acceleration is handled by migrating tensors to a `device` (e.g., CUDA for GPUs or CPU).

## 2. The Encoder Block

The Encoder consists of a stack of $N$ identical blocks. Each block contains three primary sub-modules:

### A. Input Processing (Embeddings & Positional Encoding)

Before entering the attention mechanism, raw text is transformed into continuous vectors.

1. **Tokenization:** Text is split into tokens using algorithms like Byte Pair Encoding (BPE), word-level, or sentence-level tokenizers.
2. **Input Embedding:** A trainable lookup table (`nn.Embedding`) maps each token ID in the vocabulary to a dense vector of size $d_{model}$ (the embedding dimension).
3. **Positional Encoding:** Because Transformers process tokens in parallel (unlike RNNs), they have no inherent sense of sequence order. Positional encodings are added to the input embeddings.
* **Static/Absolute (Sinusoidal):** Predefined alternating sine and cosine functions. It considers the absolute position (index 0, 1, 2) of the token.
* **Trainable (Dynamic):** A learned embedding matrix representing positions, updated during backpropagation (used in this implementation).
* **Relative:** Computes the distance between tokens rather than their absolute positions.



### B. Multi-Head Self-Attention (MHSA)

This layer allows the model to weigh the importance of other tokens in the sequence when encoding a specific token. In the Encoder, the **Query (Q), Key (K), and Value (V)** matrices all originate from the *same* previous layer's output.

### C. Add & Norm (Residual Connections & Layer Normalization)

After MHSA (and the FFN), a residual connection bypasses the sub-layer and is added to the output, followed by Layer Normalization.

* **Formula:** $\text{LayerNorm}(x + \text{Sublayer}(x))$
* *Note on transcription:* The lecture audio refers to this as "add and not" / "near zero"—the correct technical terms are **Add & Norm** and **Layer Normalization**.

### D. Feed-Forward Network (FFN)

A simple two-layer Multi-Layer Perceptron (MLP) applied to each token position independently.

* **Architecture:** Linear Layer $\rightarrow$ ReLU Activation $\rightarrow$ Linear Layer.
* **Dimensionality:** It utilizes an expansion factor (typically 4). It projects the embedding dimension from $d_{model}$ up to $4 \times d_{model}$, and then back down to $d_{model}$.
* **Dropout:** Applied after the FFN for regularization to prevent overfitting.

---

## 3. Deep Dive: Multi-Head Self-Attention Mathematics & Tensors

Understanding tensor dimensions during MHSA is critical for NPTEL numericals and implementation debugging.

### Dimensionality Setup

* $N$: Batch size (number of sequences).
* $d_{model}$: Total embedding size (`embed_size`).
* $h$: Number of attention heads (`heads`).
* $d_k, d_v$: Dimension of each head (`head_dim`).
* **Strict Assertion:** $d_{model}$ must be perfectly divisible by $h$. Thus, $d_k = \frac{d_{model}}{h}$.

### Tensor Reshaping and `einsum`

Standard matrix multiplication (`torch.matmul`) struggles with multi-dimensional batched tensors where heads need to be isolated. PyTorch's `torch.einsum` provides a generalized way to multiply tensors by specifying dimension mappings.

1. **Linear Projections:** Input $X$ is passed through three trainable weight matrices ($W^Q, W^K, W^V$) to generate $Q, K, V$.
2. **Reshaping for Heads:** The dimension $d_{model}$ is split into $(h, d_k)$.
* $Q$ shape: $(N, Q_{len}, h, d_k)$
* $K$ shape: $(N, K_{len}, h, d_k)$
* $V$ shape: $(N, V_{len}, h, d_k)$
* *Constraint:* $K_{len}$ always equals $V_{len}$.



### Calculating Attention Scores

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

1. **Dot Product ($Q K^T$):** Matches queries to keys.
* `einsum` mapping: `"nqhd, nkhd -> nhqk"`
* Resulting shape: $(N, h, Q_{len}, K_{len})$. This is the raw **attention score matrix**.


2. **Scaling:** Divided by $\sqrt{d_k}$ (square root of the head dimension) to stabilize gradients during backpropagation.
3. **Masking:** (See Section 5). Padded tokens are replaced with a large negative number ($-\infty$).
4. **Softmax:** Applied along the last dimension (dimension 3: $K_{len}$). This converts scores into a probability distribution (summing to 1) for every query over all keys.
5. **Multiply by Value ($V$):**
* `einsum` mapping: `"nhqk, nlhd -> nqhd"` (Note: $L$ represents $V_{len}$ which equals $K_{len}$).
* Resulting shape: $(N, Q_{len}, h, d_k)$.


6. **Concatenation & Final Output:** The $h$ and $d_k$ dimensions are reshaped (concatenated) back to $d_{model}$. Passed through a final linear projection matrix ($W^O$ or `FC_out`).

---

## 4. The Decoder Block

The Decoder stacks $N$ identical layers, but introduces modifications to handle sequential generation.

1. **Masked Multi-Head Self-Attention:** The first attention block. Q, K, and V all come from the Decoder's inputs. A **Target Mask** prevents tokens from attending to future positions.
2. **Cross-Attention (Encoder-Decoder Attention):**
* **Query (Q):** Comes from the previous layer of the *Decoder*.
* **Key (K) & Value (V):** Come from the final output of the *Encoder*.
* This allows the decoder to "focus" on relevant parts of the input sequence.


3. **Feed-Forward Network:** Identical structure to the Encoder's FFN.
4. **Output Projection:** The final output of dimension $d_{model}$ is passed through a linear layer (`fc_out`) projecting it to the `vocab_size`. A softmax over this vocabulary predicts the probability of the next token.

---

## 5. Masking Mechanisms

Masking prevents the attention mechanism from calculating scores for invalid or forbidden tokens by setting their raw attention scores to $-\infty$ (which becomes $0$ after Softmax).

| Mask Type | Used In | Purpose | PyTorch Implementation |
| --- | --- | --- | --- |
| **Source/Padding Mask** | Encoder & Decoder | Prevents the model from paying attention to `<PAD>` tokens added to make batch sequences equal in length. | Boolean mask where `0` indicates a pad token. |
| **Target/Look-ahead Mask** | Decoder (First Attention Block) | Prevents the autoregressive decoder from "seeing the future" (i.e., attending to tokens that haven't been generated yet). | Uses `torch.tril()` to create a lower triangular matrix. Upper triangle is masked with $-\infty$. |

*(Note on transcription: "trill" in the lecture refers to `torch.tril`, which isolates the lower triangle of a matrix).*

---

## Practice Questions (NPTEL-Style)

**Q1. A Transformer model is configured with a batch size of 32, a sequence length of 512, an embedding dimension of 768, and 12 attention heads. What is the shape of the raw attention score matrix (before multiplying by the Value matrix) for a single Multi-Head Attention layer?**
A) `(32, 512, 768)`
B) `(32, 12, 512, 512)`
C) `(32, 12, 512, 64)`
D) `(32, 512, 12, 64)`
*Step-by-step derivation:*

1. The attention score matrix results from $Q \times K^T$.
2. The `einsum` mapping is `"nqhd, nkhd -> nhqk"`.
3. $N = 32$ (batch size). $h = 12$ (heads). $Q_{len} = 512$. $K_{len} = 512$.
4. The output shape is $(N, h, Q_{len}, K_{len}) = (32, 12, 512, 512)$.
*Correct Answer: B*

**Q2. During the implementation of the Feed-Forward Network (FFN) inside a Transformer block, what is the standard dimensionality expansion and projection process if the input vector has dimension $d_{model}$?**
A) $d_{model} \rightarrow 2 \times d_{model} \rightarrow d_{model}$
B) $d_{model} \rightarrow 4 \times d_{model} \rightarrow d_{model}$
C) $d_{model} \rightarrow d_{model} / 2 \rightarrow d_{model}$
D) $d_{model} \rightarrow d_{model} / 4 \rightarrow d_{model}$
*Explanation:* The original paper and standard PyTorch implementations use an expansion factor of 4. The first linear layer up-projects to $4d$, a ReLU is applied, and the second linear layer down-projects back to $d$.
*Correct Answer: B*

**Q3. In the Decoder's cross-attention module, where do the Query (Q), Key (K), and Value (V) tensors originate?**
A) Q, K, V all come from the Decoder's input.
B) Q, K, V all come from the Encoder's output.
C) Q comes from the Encoder; K and V come from the Decoder.
D) Q comes from the Decoder; K and V come from the Encoder's output.
*Explanation:* Cross-attention bridges the encoder and decoder. The decoder generates the Query based on what it has produced so far, while the Keys and Values are the contextualized representations from the Encoder.
*Correct Answer: D*

**Q4. Why is `torch.tril()` (a lower triangular matrix) used when creating the Target Mask in the Decoder?**
A) To ignore padding tokens at the end of the sequence.
B) To ensure the embedding dimension is divisible by the number of heads.
C) To prevent a token at position $i$ from calculating attention scores with any token at position $> i$.
D) To scale the dot products before applying the Softmax function.
*Explanation:* The lower triangular matrix keeps the diagonal and everything below it (tokens up to the current step) as $1$, while the upper triangle (future tokens) becomes $0$ (mapped to $-\infty$). This ensures the model cannot "see the future" during training.
*Correct Answer: C*

---

## Quick Revision Guide

* **Embeddings & Positional Encoding:** Raw tokens $\rightarrow$ Lookup Table ($d_{model}$) + Positional Encoding (Absolute, Dynamic, or Relative).
* **MHSA Parameters:** $d_k = d_{model} / h$. $d_{model}$ must perfectly divide by the number of heads $h$.
* **einsum for Attention:** Replaces standard matmul for batched multi-head operations. Essential mapping for scores: `nqhd, nkhd -> nhqk`.
* **Attention Formula:** Requires Q, K, V. Dot product Q and K, scale by $\sqrt{d_k}$, apply mask, apply softmax across keys, multiply by V.
* **Encoder Masking:** Only uses **padding masks** (removes `<PAD>` influence).
* **Decoder Masking:** Uses **look-ahead/target masks** (lower triangular) in self-attention to hide future tokens, and **padding masks** in cross-attention.
* **Feed-Forward Architecture:** Linear $\rightarrow$ ReLU $\rightarrow$ Linear. Expands dimension by 4x internally.
* **Add & Norm:** Every sub-layer output is added to its original input (residual) and then normalized using LayerNorm.